# quantbench_paper_soxl_v1

Notebook-first evidence surface for adapting QuantBench (arXiv:2504.18600) to the SOXL harness.

The architecture logic lives in `src.soxl_quant.app.quantbench_screen`; this notebook only executes that logic and records the resulting evidence.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_paper_soxl_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.quantbench_screen import QuantBenchGate, build_quantbench_screen_report

print("root=.")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/quantbench_paper_soxl_v1


## Execute Src-Owned QuantBench Screen

The strict gate encodes the user target: full test-proxy MaxDD <= 20%, positive proxy return, strong 2022 benchmark-relative defense, nonzero stress exposure so the strategy retains chips, and at least 50% capture of 2023-2024 buy-and-hold upside.

In [2]:
gate = QuantBenchGate(
    max_drawdown_pct=20.0,
    min_stress_relative_return=30.0,
    min_stress_avg_pos=0.10,
    max_stress_avg_pos=0.45,
    min_bull_capture=0.50,
    min_proxy_return=0.0,
)
report = build_quantbench_screen_report(root=ROOT, card_dir=CARD, gate=gate)
print(json.dumps({
    "decision": report["decision"],
    "records": report["screen"]["records"],
    "eligible_profiles": report["screen"]["eligible_profiles"],
}, indent=2))

{
  "decision": "no_current_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 26,
  "eligible_profiles": []
}


## Paper Requirement Matrix

QuantBench is a full benchmark architecture: data preparation, factor mining, modeling, portfolio/execution objectives, and task-aware evaluation. The requirement matrix is emitted by src and persisted as a sidecar CSV.

In [3]:
requirements_df = pd.read_csv(CARD / "paper_requirement_matrix.csv")
display(requirements_df)

,key,paper_basis,soxl_mapping,next_artifact
0,data_preparation,"Unified data preparation with breadth, depth, ...","SOXL/SOXX/VIX daily panel, qlib binary store, ...",runtime/data or runtime/model_selection notebo...
1,factor_mining,Formula-based factor mining and learned factor...,"Feature pack diagnostics with IC, standalone b...",runtime/factor/card/<pack>/<pack>.ipynb plus f...
2,modeling_tree,Tree models can outperform deep nets on strong...,Add non-RL tabular baselines such as XGBoost/L...,runtime/model_selection/card/quantbench_paper_...
3,modeling_sequence,Temporal models such as LSTM are benchmarked a...,Existing PPO actor surfaces include MLP fallba...,network/profile matrix and smoke/full retrain ...
4,modeling_relational,Relational or graph models test whether market...,Single-SOXL setting maps this to SOXL/SOXX/VIX...,SOXX-relative and multi-asset relation feature...
5,training_objectives,"Classification, IC, MSE, ranking, utility, and...","Compare return-prediction labels, IC/ranking p...",objective-family screen with fixed-split and w...
6,rolling_continual_learning,Rolling updates mitigate alpha decay and distr...,Use existing walk-forward folds plus 3/6/12-mo...,runtime/model_selection rolling cadence notebo...
7,ensemble_robustness,Seed ensembles reduce low signal-to-noise over...,Aggregate low-correlated actors/signals only a...,ensemble candidate summary with pairwise corre...
8,evaluation_metrics,"Evaluation should include signal, portfolio, e...","Gate on IC/ICIR, return, Sharpe, MaxDD, turnov...",quantbench_screen summary.json plus candidate ...


## Model / Objective Coverage Ledger

This ledger is a coverage claim, not a performance claim. Missing families are explicit implementation tasks.

In [4]:
coverage_df = pd.read_csv(CARD / "paper_model_objective_coverage.csv")
display(coverage_df)

,family,paper_family,current_soxl_surface,coverage,next_step
0,tree_xgboost_lightgbm,XGBoost/tree model,not implemented as supervised baseline,missing,add tabular return/drawdown classifier baselin...
1,sequence_lstm_gru,LSTM/temporal sequence,GRU actor exists; LSTM supervised objective mi...,partial,compare GRU PPO actor to supervised sequence l...
2,attention_tide_mixer,deep sequence / mixer style,attention and TiDE actor surfaces exist; MLP-M...,partial,small-capacity mixer/attention screen only aft...
3,relational_rgcn,RGCN/relational model,SOXX-relative and multi-asset relation feature...,partial,screen relation features before graph implemen...
4,objectives_clf_ic_mse_ranking,CLF / IC / MSE / Ranking,PPO/BC reward objectives exist; supervised obj...,missing,implement common target builder and objective ...
5,utility_rl_execution,utility maximization / RL / execution,"PPO reward, risk budget, execution overlays exist",implemented_partial,use QuantBench gate for promotion selection
6,rolling_continual,3/6/12 month rolling updates,walk-forward folds exist; cadence-specific rol...,partial,add rolling cadence screen with cost/evidence ...
7,seed_ensemble,seed ensemble for overfit mitigation,single-profile summaries exist; no general ens...,missing,rank low-correlation candidates and ensemble o...


## Objective Family Mapping

This table is emitted by `src.soxl_quant.app.quantbench_objectives` and defines the shared SOXL target semantics for paper-style CLF, IC, MSE, ranking, utility, and RL comparisons.

In [5]:
objective_df = pd.read_csv(CARD / "paper_objective_families.csv")
display(objective_df)

,key,paper_basis,soxl_target,metric
0,clf,Classification loss predicts future up/down mo...,fwd_positive_h,clf_accuracy
1,ic,IC objective aligns predictions with future re...,fwd_return_h,ic
2,mse,MSE loss regresses future returns directly.,fwd_return_h,mse
3,ranking,Pairwise or listwise ranking objectives prefer...,relative ordering of fwd_return_h,ranking_spread
4,utility,Utility maximization should reflect trading ou...,position-weighted fwd_return_h with chip reten...,utility_return
5,rl,Non-differentiable simulation and execution fe...,PPO reward and execution gate metrics,portfolio_return_sharpe_maxdd


## Current Runtime Candidate Screen

The ranking scans current runtime summaries through the src-owned QuantBench gate. It does not retrain models in this notebook.

In [6]:
screen_df = pd.read_csv(CARD / "candidate_screen.csv")
print({"eligible": int(screen_df["eligible"].sum()) if not screen_df.empty else 0, "rows": len(screen_df)})
display(screen_df.head(20))

{'eligible': 0, 'rows': 26}


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,balanced_rule_residual_v19_reg_v22_retest_v1,runtime/model/card/balanced_rule_residual_v19_...,False,1119.521092,"['max_drawdown', 'stress_max_drawdown']",1280.416932,59.637288,37.652465,58.299843,0.380930,44.524050,55.654474,0.800008
1,balanced_rule_residual_v22_continued_retention_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1015.381197,"['max_drawdown', 'stress_max_drawdown']",1128.935617,59.130395,53.736876,55.255626,0.321447,60.791539,55.654474,1.092303
2,balanced_rule_residual_v22_release_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1002.849540,"['max_drawdown', 'stress_max_drawdown']",1133.887381,60.350126,45.981620,38.845207,0.375263,58.092719,55.654474,1.043810
3,balanced_rule_residual_v19_moe_v2,runtime/model/card/balanced_rule_residual_v19_...,False,610.398973,"['max_drawdown', 'stress_max_drawdown']",690.971221,49.008279,47.586050,45.149295,0.287632,60.037161,55.654474,1.078748
4,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,559.292883,"['max_drawdown', 'stress_max_drawdown']",619.708826,43.565118,52.906964,41.483544,0.258405,56.176271,55.654474,1.009376
5,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,426.778414,"['max_drawdown', 'stress_max_drawdown']",496.780712,44.234213,65.015154,43.909386,0.131381,45.591811,55.654474,0.819194
6,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,351.838655,"['max_drawdown', 'stress_max_drawdown']",392.580698,37.676486,55.990314,35.878546,0.185465,52.300504,55.654474,0.939736
7,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,-1032.939429,"['proxy_return', 'bull_capture']",-7.000281,9.875679,53.563568,9.875679,0.165855,NaN,NaN,NaN
8,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,-1054.613515,"['proxy_return', 'bull_capture']",-12.705282,15.900697,61.694557,15.900697,0.141487,NaN,NaN,NaN
9,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,-1174.741020,"['max_drawdown', 'proxy_return', 'stress_max_d...",-23.275008,29.147985,45.125927,29.147985,0.243813,NaN,NaN,NaN


## Conclusion

This notebook executes the first src-owned QuantBench-to-SOXL architecture layer: requirement coverage, model/objective coverage, runtime summary discovery, and strict candidate selection for MaxDD < 20%, stress chip retention, and bull-market buy-and-hold capture.

The current screen is conservative. If `summary.json` reports `no_current_model_satisfies_quantbench_soxl_20dd_gate`, existing runtime summaries do not yet prove the requested end state. The next implementation step is to add the missing supervised tabular/sequence objective screen from the paper and evaluate it through the same src-owned gate before attempting heavier network additions or ensembles.